# POIデータで街をクラスタリング

東京都内656駅周辺の「1地域＝1行」の特徴量を、特定の街を基準にせず分類します。前回と同じ固定入力を使い、各駅中心1km四方のPOI登録を比較します。

主結果は3群ですが、明確に分離した街種ではありません。seedには安定し、重複AOIを抑制した学習には依存しました。街の優劣・住みやすさ・商圏を示しません。

[方法](../docs/clustering.md) / [検証](../docs/clustering-validation.md) / [kの選択](../docs/clustering-selection.md)。事前に手順に従って環境と保存済み入力を用意してください。

## 1. 検証済み特徴量を読む
入力のハッシュ・656行・駅IDの対応を確認します。密度と集中度のlog1p、6カテゴリの重みは前回と共通です。

In [ ]:
from geoai_open_lab.region_clustering import load_inputs, run_clustering, ROOT
import pandas as pd
import json
from IPython.display import SVG, display
profiles, stations, source, config = load_inputs()
print(f"{len(profiles)}駅 × {len(profiles.columns)}列（うち8列を分類へ）")

## 2. クラスタ数を比較する
k=3〜8を評価し、seed=0〜9とWard法も比較します。PCAへ落とす前の8次元で計算します。

In [ ]:
presentation = json.loads((ROOT / "configs/clustering-presentation.json").read_text(encoding="utf8"))
metrics, manifest, output_dir = run_clustering(k=presentation["main_k"])
metrics[["k", "silhouette", "calinski_harabasz", "davies_bouldin", "min_cluster_size", "seed_pair_ari_min"]]

## 3. 主結果は3群
全群165駅以上で、seed安定性と説明しやすさを優先しました。silhouette最大のk=4、DB最小のk=5も記録しています。k=4/5の衣料比率が高い15駅群は、この粗い分類では見えにくくなります。

In [ ]:
summary = pd.read_csv(output_dir / "cluster_summary.csv")
summary[["cluster", "count", "representatives", "contrasting_station"]]

## 4. 数値プロフィールを見る
生の中央値を確認してから、全駅平均との差を図にします。代表駅は特徴空間の重心近傍で、地理的中心ではありません。

In [ ]:
details = pd.read_csv(output_dir / "cluster_features.csv")
details.pivot(index="feature", columns="cluster", values="median").round(4)

## 5. 図で比較する
ヒートマップは重み付け後の平均差。地図は駅代表点、PCAは表示専用です。

In [ ]:
from geoai_open_lab.clustering_figures import save_clustering_figures
figures = save_clustering_figures(output_dir, presentation["names"])
for name in ["heatmap", "station_map", "pca", "cluster_sizes", "k_comparison"]:
    display(SVG(filename=str(output_dir / f"{name}.svg")))

## 6. 安定性と限界
seed間ARI最小0.975でも、非重複313駅での再学習とのARIは0.271です。全656駅は独立した街ではありません。POI10件で構成比が高くなる駅もあり、群名を全駅へ当てはめないでください。

In [ ]:
pd.Series({"WardとのARI": manifest["ward_ari"],
           "重複抑制版とのARI（全駅）": manifest["suppressed_all_ari"],
           "PCA2軸の累積寄与率": sum(manifest["pca_explained_ratio"])})

今回選んだPOI特徴の分類であり、人出・人口・景観・住みやすさは測っていません。地域ごとの収録差・欠損も残ります。次回はRegion Embeddingが元特徴の近傍関係や分類をどう変えるかを検証する予定です。PCAの可視化だけでEmbeddingを評価したことにはしません。